# Simulazione e analisi statistica di una coda

## Confronto tra servizio esponenziale e servizio uniforme

### Domanda del progetto

A parita' di tempo medio di servizio, come cambiano il tempo di attesa e il numero di clienti in coda se la durata del servizio segue una distribuzione esponenziale oppure uniforme?

Il progetto utilizza soltanto argomenti presenti nelle lezioni:

- distribuzioni di Poisson, esponenziale e uniforme;
- numeri pseudocasuali e simulazione Monte Carlo;
- media, mediana e deviazione standard;
- probabilita' frequentista ed eventi;
- istogrammi e densita' di probabilita';
- test di Kolmogorov-Smirnov;
- regressione lineare e coefficiente di correlazione.

Non vengono utilizzate formule specifiche esterne della teoria delle code.


## 1. Librerie

Sono le librerie utilizzate nelle lezioni per il calcolo numerico, le tabelle, i grafici e le distribuzioni di probabilita'.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from scipy.stats import expon
from scipy.stats import uniform
from scipy import stats

np.random.seed(1234)


## 2. Ipotesi e parametri

Si considera una coda con una sola cassa. Arrivano mediamente 10 clienti ogni ora e ogni servizio dura mediamente 4 minuti.

Gli intervalli fra due arrivi consecutivi seguono una distribuzione esponenziale. Questa ipotesi corrisponde a un processo di arrivi poissoniano, studiato nelle lezioni.

Si confrontano due modelli per la durata del servizio:

1. distribuzione esponenziale con media 4 minuti;
2. distribuzione uniforme tra 2 e 6 minuti, ancora con media 4 minuti.

Tutto resta uguale nei due esperimenti, tranne la distribuzione del servizio.


In [ ]:
tasso_arrivi = 10 / 60
tempo_medio_servizio = 4.0

n_clienti = 20000
n_da_scartare = 1000

print("Arrivi medi ogni ora:", tasso_arrivi * 60)
print("Tempo medio di servizio:", tempo_medio_servizio, "minuti")
print("Clienti simulati:", n_clienti)


## 3. Funzione di simulazione

Per ogni cliente vengono calcolati:

- istante di arrivo;
- durata del servizio;
- istante di inizio del servizio;
- istante di fine del servizio;
- tempo di attesa;
- numero di clienti trovati in coda.

I clienti vengono serviti nell'ordine di arrivo.


In [ ]:
def simula_coda(n, tasso_arrivi, tempo_medio_servizio, tipo_servizio):
    intervalli_arrivo = expon.rvs(scale=1/tasso_arrivi, size=n)

    tempi_arrivo = np.zeros(n)
    for i in range(1, n):
        tempi_arrivo[i] = tempi_arrivo[i-1] + intervalli_arrivo[i]

    if tipo_servizio == 'esponenziale':
        tempi_servizio = expon.rvs(scale=tempo_medio_servizio, size=n)
    else:
        tempi_servizio = uniform.rvs(loc=2, scale=4, size=n)

    inizio_servizio = np.zeros(n)
    fine_servizio = np.zeros(n)
    tempi_attesa = np.zeros(n)
    numero_in_coda = np.zeros(n, dtype=int)

    primo_non_uscito = 0

    for i in range(n):
        if i == 0:
            inizio_servizio[i] = tempi_arrivo[i]
        else:
            inizio_servizio[i] = max(tempi_arrivo[i], fine_servizio[i-1])

        tempi_attesa[i] = inizio_servizio[i] - tempi_arrivo[i]
        fine_servizio[i] = inizio_servizio[i] + tempi_servizio[i]

        while (primo_non_uscito < i and
               fine_servizio[primo_non_uscito] <= tempi_arrivo[i]):
            primo_non_uscito = primo_non_uscito + 1

        persone_presenti = i - primo_non_uscito
        numero_in_coda[i] = max(persone_presenti - 1, 0)

    dati = pd.DataFrame({
        'arrivo': tempi_arrivo,
        'servizio': tempi_servizio,
        'attesa': tempi_attesa,
        'numero_in_coda': numero_in_coda
    })

    return dati


## 4. Esecuzione delle due simulazioni

Si eliminano i primi 1000 clienti per ridurre l'effetto iniziale dovuto al fatto che la simulazione parte con la cassa vuota.


In [ ]:
dati_exp = simula_coda(n_clienti, tasso_arrivi,
                       tempo_medio_servizio, 'esponenziale')

dati_uni = simula_coda(n_clienti, tasso_arrivi,
                       tempo_medio_servizio, 'uniforme')

dati_exp = dati_exp.iloc[n_da_scartare:]
dati_uni = dati_uni.iloc[n_da_scartare:]

print("Clienti analizzati per ogni modello:", len(dati_exp))
dati_exp.head()


## 5. Controllo delle distribuzioni del servizio

Prima di analizzare la coda si controlla che i tempi generati seguano effettivamente le distribuzioni scelte.

Per il servizio esponenziale la densita' teorica e'

$$f(t)=\frac{1}{\tau}e^{-t/\tau},$$

con $\tau=4$ minuti. Il servizio uniforme e' invece compreso tra 2 e 6 minuti.


In [ ]:
x = np.linspace(0, 20, 500)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(dati_exp.servizio, bins=50, density=True, alpha=0.6,
         label='Dati simulati')
plt.plot(x, expon.pdf(x, scale=4), color='red',
         label='Densita teorica')
plt.xlabel('Tempo di servizio [min]')
plt.ylabel('Densita di probabilita')
plt.title('Servizio esponenziale')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(dati_uni.servizio, bins=40, density=True, alpha=0.6,
         label='Dati simulati')
plt.plot(x, uniform.pdf(x, loc=2, scale=4), color='red',
         label='Densita teorica')
plt.xlim(0, 8)
plt.xlabel('Tempo di servizio [min]')
plt.ylabel('Densita di probabilita')
plt.title('Servizio uniforme')
plt.legend()

plt.tight_layout()
plt.show()


### Test di Kolmogorov-Smirnov

L'ipotesi nulla del test e' che il campione provenga dalla distribuzione teorica indicata. Se il p-value non e' piccolo, i dati simulati risultano compatibili con il modello scelto.


In [ ]:
D_exp, p_exp = stats.kstest(dati_exp.servizio,
                            expon(scale=4).cdf)

D_uni, p_uni = stats.kstest(dati_uni.servizio,
                            uniform(loc=2, scale=4).cdf)

print("KS esponenziale: D = {:.4f}, p-value = {:.4f}".format(D_exp, p_exp))
print("KS uniforme: D = {:.4f}, p-value = {:.4f}".format(D_uni, p_uni))


## 6. Statistica descrittiva dei tempi di attesa

Per i due modelli vengono confrontati:

- attesa media;
- attesa mediana;
- deviazione standard;
- probabilita' frequentista di aspettare piu' di 5 minuti;
- numero medio di clienti in coda.


In [ ]:
risultati = pd.DataFrame({
    'modello': ['Esponenziale', 'Uniforme'],
    'attesa_media': [dati_exp.attesa.mean(), dati_uni.attesa.mean()],
    'attesa_mediana': [dati_exp.attesa.median(), dati_uni.attesa.median()],
    'deviazione_standard': [dati_exp.attesa.std(), dati_uni.attesa.std()],
    'probabilita_oltre_5_min': [
        (dati_exp.attesa > 5).mean(),
        (dati_uni.attesa > 5).mean()
    ],
    'numero_medio_in_coda': [
        dati_exp.numero_in_coda.mean(),
        dati_uni.numero_in_coda.mean()
    ]
})

risultati


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(dati_exp.attesa, bins=70, range=(0, 35), density=True,
         alpha=0.5, label='Servizio esponenziale')
plt.hist(dati_uni.attesa, bins=70, range=(0, 35), density=True,
         alpha=0.5, label='Servizio uniforme')
plt.xlabel('Tempo di attesa [min]')
plt.ylabel('Densita di probabilita')
plt.title('Distribuzione dei tempi di attesa')
plt.legend()
plt.show()


## 7. Distribuzione del numero di clienti in coda

Per ogni possibile numero di persone in coda viene calcolata la frequenza relativa osservata nella simulazione.


In [ ]:
valori_coda = np.arange(0, 11)
frequenze_exp = []
frequenze_uni = []

for n in valori_coda:
    frequenze_exp.append((dati_exp.numero_in_coda == n).mean())
    frequenze_uni.append((dati_uni.numero_in_coda == n).mean())

plt.figure(figsize=(9, 5))
plt.bar(valori_coda-0.2, frequenze_exp, width=0.4,
        label='Servizio esponenziale')
plt.bar(valori_coda+0.2, frequenze_uni, width=0.4,
        label='Servizio uniforme')
plt.xlabel('Numero di clienti in coda')
plt.ylabel('Frequenza relativa')
plt.title('Distribuzione del numero di clienti in coda')
plt.xticks(valori_coda)
plt.legend()
plt.show()


## 8. Probabilita' di eventi e loro intersezione

Nel caso del servizio esponenziale si definiscono due eventi:

$$A=\{\text{il cliente aspetta piu' di 5 minuti}\},$$

$$B=\{\text{il cliente trova almeno 2 persone in coda}\}.$$

L'intersezione $A\cap B$ contiene i clienti per i quali si verificano entrambe le condizioni.

Le probabilita' vengono stimate in senso frequentista, dividendo il numero di casi favorevoli per il numero totale di clienti simulati.


In [ ]:
evento_A = dati_exp.attesa > 5
evento_B = dati_exp.numero_in_coda >= 2
intersezione_AB = evento_A & evento_B
unione_AB = evento_A | evento_B

prob_A = evento_A.mean()
prob_B = evento_B.mean()
prob_intersezione = intersezione_AB.mean()
prob_unione = unione_AB.mean()

print("P(A) = {:.4f}".format(prob_A))
print("P(B) = {:.4f}".format(prob_B))
print("P(A intersezione B) = {:.4f}".format(prob_intersezione))
print("P(A unione B) = {:.4f}".format(prob_unione))
print("Controllo unione = {:.4f}".format(prob_A + prob_B - prob_intersezione))


### Rappresentazione degli eventi nel quadrato

Il quadrato rappresenta lo spazio degli eventi. Il rosso rappresenta A, il blu rappresenta B e la zona viola sovrapposta rappresenta l'intersezione. Le aree del disegno sono illustrative: le probabilita' numeriche corrette sono quelle calcolate dalla simulazione e scritte nel grafico.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

spazio = patches.Rectangle((0, 0), 1, 1,
                           linewidth=2, edgecolor='black',
                           facecolor='white')

rettangolo_A = patches.Rectangle((0.10, 0.20), 0.55, 0.65,
                                 linewidth=2, edgecolor='red',
                                 facecolor='red', alpha=0.35)

rettangolo_B = patches.Rectangle((0.35, 0.08), 0.55, 0.65,
                                 linewidth=2, edgecolor='blue',
                                 facecolor='blue', alpha=0.35)

ax.add_patch(spazio)
ax.add_patch(rettangolo_A)
ax.add_patch(rettangolo_B)

plt.text(0.18, 0.78, 'A', fontsize=18, color='red')
plt.text(0.82, 0.15, 'B', fontsize=18, color='blue')
plt.text(0.43, 0.45, 'A intersezione B', fontsize=11)

plt.text(0.03, -0.10,
         'P(A)={:.3f}   P(B)={:.3f}   P(A intersezione B)={:.3f}'
         .format(prob_A, prob_B, prob_intersezione), fontsize=10)

plt.xlim(-0.05, 1.05)
plt.ylim(-0.15, 1.05)
plt.xlabel('Spazio degli eventi')
plt.ylabel('Spazio degli eventi')
plt.title('Eventi A, B e loro intersezione')
plt.show()


## 9. Modello quantitativo con regressione lineare

Si studia la relazione tra:

- variabile indipendente $x$: numero di clienti trovati in coda;
- variabile dipendente $y$: tempo di attesa.

Si propone il modello lineare

$$y=a+bx.$$

La regressione stima intercetta $a$ e coefficiente angolare $b$. Il coefficiente di correlazione $r$ misura l'intensita' della relazione lineare, mentre $r^2$ indica quale parte della variabilita' dell'attesa e' descritta dal modello.

Questa analisi utilizza la regressione lineare affrontata nelle lezioni e costituisce il modello statistico proposto per i dati simulati.


In [ ]:
x_reg = dati_exp.numero_in_coda
y_reg = dati_exp.attesa

regressione = stats.linregress(x_reg, y_reg)

pendenza = regressione.slope
intercetta = regressione.intercept
r = regressione.rvalue
r_quadro = r**2
p_value_regressione = regressione.pvalue

print("Intercetta = {:.3f}".format(intercetta))
print("Pendenza = {:.3f} minuti per cliente in coda".format(pendenza))
print("Coefficiente di correlazione r = {:.3f}".format(r))
print("Coefficiente r^2 = {:.3f}".format(r_quadro))
print("p-value della regressione = {:.4e}".format(p_value_regressione))


In [ ]:
x_linea = np.arange(0, 11)
y_linea = intercetta + pendenza * x_linea

attesa_media_per_coda = []

for n in x_linea:
    selezione = dati_exp[dati_exp.numero_in_coda == n]
    attesa_media_per_coda.append(selezione.attesa.mean())

plt.figure(figsize=(9, 5))
plt.plot(x_linea, attesa_media_per_coda, 'o',
         label='Medie della simulazione')
plt.plot(x_linea, y_linea, color='red',
         label='Retta di regressione')
plt.xlabel('Numero di clienti trovati in coda')
plt.ylabel('Tempo medio di attesa [min]')
plt.title('Relazione tra lunghezza della coda e attesa')
plt.legend()
plt.show()


## 10. Ripetizioni Monte Carlo

Una sola simulazione puo' fornire una media casualmente piu' alta o piu' bassa. L'esperimento viene quindi ripetuto 100 volte. Si studia la distribuzione delle medie dei tempi di attesa ottenute nelle diverse simulazioni.


In [ ]:
def ripeti_simulazioni(n_ripetizioni, n_clienti_mc, tipo_servizio):
    medie = []

    for i in range(n_ripetizioni):
        dati = simula_coda(n_clienti_mc, tasso_arrivi,
                           tempo_medio_servizio, tipo_servizio)
        dati = dati.iloc[500:]
        medie.append(dati.attesa.mean())

    return np.array(medie)


In [ ]:
n_ripetizioni = 100

medie_exp = ripeti_simulazioni(n_ripetizioni, 5000,
                               'esponenziale')

medie_uni = ripeti_simulazioni(n_ripetizioni, 5000,
                               'uniforme')

print("Media delle medie - esponenziale = {:.3f}".format(medie_exp.mean()))
print("Deviazione standard delle medie - esponenziale = {:.3f}".format(medie_exp.std()))
print("Media delle medie - uniforme = {:.3f}".format(medie_uni.mean()))
print("Deviazione standard delle medie - uniforme = {:.3f}".format(medie_uni.std()))


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(medie_exp, bins=20, alpha=0.5,
         label='Servizio esponenziale')
plt.hist(medie_uni, bins=20, alpha=0.5,
         label='Servizio uniforme')
plt.xlabel('Tempo medio di attesa [min]')
plt.ylabel('Numero di simulazioni')
plt.title('Distribuzione delle medie nelle ripetizioni Monte Carlo')
plt.legend()
plt.show()


## 11. Studio al variare del tasso di arrivo

Si ripete infine la simulazione con 6, 10 e 13 arrivi medi ogni ora. In questo modo si osserva sperimentalmente come cambiano il tempo medio di attesa e il numero medio di clienti in coda quando aumentano gli arrivi.

Non viene imposta una formula esterna: i risultati vengono ricavati dai dati simulati.


In [ ]:
arrivi_ogni_ora = [6, 10, 13]
studio_arrivi = []

for arrivi in arrivi_ogni_ora:
    lam = arrivi / 60

    dati = simula_coda(30000, lam, tempo_medio_servizio,
                       'esponenziale')
    dati = dati.iloc[1000:]

    studio_arrivi.append({
        'arrivi_ora': arrivi,
        'attesa_media': dati.attesa.mean(),
        'numero_medio_in_coda': dati.numero_in_coda.mean(),
        'probabilita_oltre_5_min': (dati.attesa > 5).mean()
    })

studio_arrivi = pd.DataFrame(studio_arrivi)
studio_arrivi


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(studio_arrivi.arrivi_ora, studio_arrivi.attesa_media,
         marker='o')
plt.xlabel('Arrivi medi ogni ora')
plt.ylabel('Tempo medio di attesa [min]')
plt.title('Arrivi e tempo di attesa')

plt.subplot(1, 2, 2)
plt.plot(studio_arrivi.arrivi_ora,
         studio_arrivi.numero_medio_in_coda, marker='o')
plt.xlabel('Arrivi medi ogni ora')
plt.ylabel('Numero medio in coda')
plt.title('Arrivi e lunghezza della coda')

plt.tight_layout()
plt.show()


## 12. Conclusioni da completare dopo l'esecuzione

Nelle conclusioni bisogna commentare i valori numerici ottenuti:

1. verificare mediante istogrammi e test KS che i tempi di servizio seguano i modelli scelti;
2. confrontare media, mediana e deviazione standard delle attese;
3. confrontare la probabilita' di attendere piu' di 5 minuti;
4. confrontare la distribuzione del numero di clienti in coda;
5. interpretare le probabilita' degli eventi $A$, $B$ e $A\cap B$;
6. interpretare pendenza, correlazione, $r^2$ e p-value della regressione;
7. verificare con Monte Carlo che il confronto fra i due servizi sia stabile;
8. descrivere come cambiano attesa e coda quando aumentano gli arrivi.

### Interpretazione attesa

Pur avendo la stessa media di 4 minuti, il servizio esponenziale e' piu' variabile e puo' produrre servizi molto lunghi. La simulazione dovrebbe quindi mostrare attese e code maggiori rispetto al servizio uniforme. La regressione dovrebbe inoltre mostrare una relazione positiva tra numero di clienti in coda e tempo di attesa.

### Limiti del modello

- una sola cassa;
- tasso di arrivo costante;
- clienti serviti nell'ordine di arrivo;
- nessun cliente abbandona la coda;
- arrivi e servizi indipendenti;
- parametri ipotetici e non ricavati da una coda reale.


## Corrispondenza con le lezioni

- generatori casuali e distribuzioni: lezioni 01, 04, 05, 06 e 07;
- Pandas e analisi dei dati: lezione 03;
- indicatori statistici: lezione 04.02;
- probabilita' frequentista: lezione 05.03;
- grafico con rettangoli: stesso strumento usato nella lezione 07.03;
- Monte Carlo: lezione 08.01;
- regressione lineare: lezione 15.01;
- test di Kolmogorov-Smirnov: lezione 18.03.
